This script ingests data through API from CMS datasets for The Medicare Physician & Other Practitioners by Provider and Service dataset provides information on use, payments, and submitted charges organized by National Provider Identifier (NPI), Healthcare Common Procedure Coding System (HCPCS) code, and place of service.

The last update of the page was on May 21, 2026 when the was available through the year 2024.

In [0]:

import requests
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

import json, re, os, time
# from datetime import datetime, timezone

dbutils.widgets.text("specialty", "")
# dbutils.widgets.text("run_date", "")
dbutils.widgets.text("force_refresh", "false")

specialty = dbutils.widgets.get("specialty").strip().lower()
# run_date = dbutils.widgets.get("run_date") or datetime.now(timezone.utc).strftime("%Y-%m-%d")
force = dbutils.widgets.get("force_refresh").lower() == "true"
# assert specialty, "specialty parameter is required"

spark = SparkSession.builder.getOrCreate()

table_name = "northstar_network_bronze.cms_provider_specialty.api_ingestion_table"

cms_api_link = "https://data.cms.gov/data-api/v1/dataset/92396110-2aed-4d63-a6a2-5d6207d46a29/data"

def fetch_cms(provider_type=specialty):
    PAGE_SIZE = 5000
    records, offset = [], 0
    while True:
        resp = requests.get(
            cms_api_link,
            params={
                "filter[Rndrng_Prvdr_Type]": provider_type,  # requests URL-encodes brackets and spaces
                "size": PAGE_SIZE,
                "offset": offset,
            },
            timeout=60,
        )
        resp.raise_for_status()
        page = resp.json()
        if not page:                 # empty page = no more data
            break
        records.extend(page)
        if len(page) < PAGE_SIZE:    # short page = last page
            break
        offset += PAGE_SIZE
    return records



In [0]:
cms_data = fetch_cms(provider_type="Pathology")
cms_df = spark.createDataFrame(cms_data)
cms_df = (cms_df.withColumn("_ingest_date", F.current_date()).withColumn("_ingest_time", F.current_timestamp()))

display(cms_df)


In [0]:
cms_df.write.mode("overwrite").saveAsTable(table_name)

In [0]:
display(cms_df.count())

In [0]:
import requests
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

def ingest_cms_data(year):

    spark = SparkSession.builder.getOrCreate()

    cms_api_link = "https://data.cms.gov/data-api/v1/dataset/92396110-2aed-4d63-a6a2-5d6207d46a29/data"

    params={
        "filter[Rndrng_Prvdr_Type]": provider_type,
        }
    
    response = requests.get(cms_api_link)

    print(f"processed with Status code: {response.status_code}")

    cms_data = response.json()

    cms_df = spark.createDataFrame(cms_data)        # Column Rndrng_NPI is the unique identifier
    cms_df = (cms_df.withColumn("_ingest_date", F.current_date()).withColumn("_ingest_time", F.current_timestamp()))

    return cms_df



In [0]:
cms_df.select("Rndrng_NPI").distinct().count()
